# Class Imbalance Analysis: Experiment 3

**Objective**: Analyze how method performance degrades as class imbalance increases (minority proportion decreases).

---

## Overview

**Experiment 3** evaluates performance degradation across varying minority class proportions:
1. Tests methods at multiple minority proportions (e.g., 0.50, 0.45, ..., 0.01)
2. Uses **NO HPO** (default parameters only) to isolate the effect of class imbalance
3. Identifies which methods are most robust to severe class imbalance
4. Helps select methods appropriate for balanced vs. highly imbalanced datasets

**Why This Matters for Credit Risk:**
- Credit default datasets are typically highly imbalanced (default rates often 1-10%)
- Methods that maintain performance under severe imbalance are critical for real-world deployment
- This experiment reveals which methods need class weights, oversampling, or other techniques

This notebook provides:
- **Imbalance Curve Plots**: Performance vs. minority class proportion
- **Degradation Analysis**: How much performance drops as imbalance increases
- **Robustness Rankings**: Which methods maintain performance under severe imbalance
- **Heatmaps**: Performance matrices across minority proportions and methods

---

## 1. Setup & Configuration

In [ ]:
# Standard imports
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# =============================================================================
# SETUP PATHS (Notebook is in notebooks/ folder)
# =============================================================================

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # notebooks/ -> project root
sys.path.insert(0, str(PROJECT_ROOT))

EXPERIMENT_NAME = "experiment3"
RESULTS_DIR = PROJECT_ROOT / "results" / EXPERIMENT_NAME
SUMMARY_DIR = RESULTS_DIR / "summary"
FIGURES_DIR = RESULTS_DIR / "figures"

print("=" * 80)
print("  EXPERIMENT 3 POST-ANALYSIS: CLASS IMBALANCE ANALYSIS")
print("=" * 80)
print(f"Project root:       {PROJECT_ROOT}")
print(f"Results directory:  {RESULTS_DIR}")
print(f"Summary directory:  {SUMMARY_DIR}")
print(f"Figures directory:  {FIGURES_DIR}")

# =============================================================================
# GENERATE SUMMARIES IF NOT ALREADY PRESENT
# =============================================================================

# Check if summary folder exists and has files
if SUMMARY_DIR.exists() and any(SUMMARY_DIR.glob("*.csv")):
    print(f"\n Summary files already exist. Skipping generation.")
    print(f"   (Delete {SUMMARY_DIR} manually to regenerate)")
else:
    print(f"\n Summary files not found. Generating summaries...")
    
    # Import and run summarize_results
    from src.utils.summarize_results import summarize_results
    
    print("\n" + "=" * 80)
    summarize_results(experiment=EXPERIMENT_NAME)
    print("=" * 80)

# Ensure figures directory exists
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("\n Setup complete! Ready for analysis.")
print("=" * 80)

In [ ]:
# Plotting configuration
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_style('whitegrid')

# Color schemes
CMAP_PERFORMANCE = 'RdYlGn'       # Red (bad) to Green (good)
CMAP_DEGRADATION = 'RdYlGn_r'     # Green (low degradation) to Red (high)
CMAP_IMBALANCE = 'coolwarm'       # Blue (balanced) to Red (imbalanced)

# Use a distinct color palette for imbalance curves
N_COLORS = 20
IMBALANCE_CURVE_PALETTE = sns.color_palette('husl', n_colors=N_COLORS)

## 2. Data Loading

In [ ]:
# Load raw and aggregated results
# Note: Experiment 3 only applies to PD (classification) tasks
try:
    pd_raw = pd.read_csv(SUMMARY_DIR / "summary_pd_raw.csv")
    pd_agg = pd.read_csv(SUMMARY_DIR / "summary_pd_aggregated.csv")
    has_pd = True
except FileNotFoundError:
    print("Warning: PD results not found")
    pd_raw = pd.DataFrame()
    pd_agg = pd.DataFrame()
    has_pd = False

print("\n Data Overview:")

if has_pd and not pd_raw.empty:
    # Determine if this is a class imbalance experiment (minority_proportion column)
    if 'minority_proportion' in pd_raw.columns:
        proportions = sorted(pd_raw['minority_proportion'].unique(), reverse=True)
        print(f"\nPD (Probability of Default) - CLASS IMBALANCE ANALYSIS:")
        print(f"  - Raw results: {len(pd_raw)} fold results")
        print(f"  - Methods: {pd_raw['method'].nunique()}")
        print(f"  - Datasets: {pd_raw['dataset'].nunique()}")
        print(f"  - Minority proportions: {len(proportions)} ({max(proportions):.2f} -> {min(proportions):.2f})")
        print(f"  - Methods: {sorted(pd_raw['method'].unique())}")
        print(f"\n  Proportion sequence: {[f'{p:.2f}' for p in proportions]}")
    else:
        print("\nWARNING: Data does not appear to be from Experiment 3 (missing minority_proportion column)")
        print(f"         Columns found: {pd_raw.columns.tolist()}")

In [ ]:
# Display sample of raw data
if has_pd and not pd_raw.empty:
    print("\nSample PD Raw Data (first 10 rows):")
    display(pd_raw.head(10))
    
    print("\nPD Aggregated Data columns:")
    print(pd_agg.columns.tolist())

## 3. Helper Functions

In [ ]:
def plot_imbalance_curves(
    df: pd.DataFrame,
    metric: str,
    task_name: str,
    dataset: str = None,
    top_n: int = 15,
    figsize: tuple = (14, 8),
    show_std: bool = True
):
    """
    Plot imbalance curves showing performance vs. minority class proportion.
    
    Args:
        df: Aggregated dataframe with minority_proportion column
        metric: Metric to plot (e.g., 'AUC', 'F1')
        task_name: 'PD' or 'LGD'
        dataset: Optional specific dataset to plot
        top_n: Number of top methods to show
        figsize: Figure size
        show_std: Whether to show standard deviation bands
    """
    mean_col = f'{metric}_mean'
    std_col = f'{metric}_std'
    
    if mean_col not in df.columns:
        print(f"Warning: Metric '{metric}' not found in data")
        return None
    
    # Filter by dataset if specified
    plot_df = df.copy()
    if dataset is not None:
        plot_df = plot_df[plot_df['dataset'] == dataset]
        title_suffix = f" - {dataset}"
    else:
        # Average across all datasets
        plot_df = plot_df.groupby(['method', 'minority_proportion']).agg({
            mean_col: 'mean',
            std_col: 'mean'
        }).reset_index()
        title_suffix = " (Averaged Across Datasets)"
    
    # Get top N methods by average performance at highest minority proportion (most balanced)
    max_proportion = plot_df['minority_proportion'].max()
    top_methods = (
        plot_df[plot_df['minority_proportion'] == max_proportion]
        .nlargest(top_n, mean_col)['method']
        .tolist()
    )
    
    # Filter to top methods
    plot_df = plot_df[plot_df['method'].isin(top_methods)]
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot each method
    colors = sns.color_palette('husl', n_colors=len(top_methods))
    
    for i, method in enumerate(top_methods):
        method_df = plot_df[plot_df['method'] == method].sort_values('minority_proportion', ascending=False)
        
        x = method_df['minority_proportion'].values
        y = method_df[mean_col].values
        
        # Plot line
        ax.plot(x, y, 'o-', label=method, color=colors[i], linewidth=2, markersize=6)
        
        # Add std band if available and requested
        if show_std and std_col in method_df.columns:
            y_std = method_df[std_col].values
            ax.fill_between(x, y - y_std, y + y_std, color=colors[i], alpha=0.15)
    
    ax.set_xlabel('Minority Class Proportion', fontsize=12, fontweight='bold')
    ax.set_ylabel(f'{metric}', fontsize=12, fontweight='bold')
    ax.set_title(f'{task_name} Imbalance Curves: {metric}{title_suffix}',
                 fontsize=14, fontweight='bold', pad=15)
    
    # Invert x-axis so imbalance increases left-to-right
    ax.invert_xaxis()
    
    # Format x-axis as percentage
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))
    
    # Add annotation for direction
    ax.annotate('Balanced', xy=(ax.get_xlim()[1], ax.get_ylim()[0]), fontsize=10, alpha=0.7)
    ax.annotate('Highly Imbalanced', xy=(ax.get_xlim()[0], ax.get_ylim()[0]), fontsize=10, alpha=0.7, ha='right')
    
    # Legend outside plot
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
    
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save figure
    ds_suffix = f"_{dataset.replace('.', '_')}" if dataset else ""
    filename = FIGURES_DIR / f"{task_name.lower()}_imbalance_curve_{metric.lower()}{ds_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")
    
    return fig

In [ ]:
def plot_imbalance_heatmap(
    df: pd.DataFrame,
    metric: str,
    task_name: str,
    dataset: str = None,
    figsize: tuple = (18, 10)
):
    """
    Create a heatmap showing performance across methods and minority proportions.
    
    Args:
        df: Aggregated dataframe with minority_proportion column
        metric: Metric to visualize
        task_name: 'PD' or 'LGD'
        dataset: Optional specific dataset
        figsize: Figure size
    """
    mean_col = f'{metric}_mean'
    
    if mean_col not in df.columns:
        print(f"Warning: Metric '{metric}' not found")
        return None
    
    # Filter by dataset if specified
    plot_df = df.copy()
    if dataset is not None:
        plot_df = plot_df[plot_df['dataset'] == dataset]
        title_suffix = f" - {dataset}"
    else:
        # Average across datasets
        plot_df = plot_df.groupby(['method', 'minority_proportion'])[mean_col].mean().reset_index()
        title_suffix = " (Averaged)"
    
    # Create pivot table: methods (rows) x minority_proportions (columns)
    pivot = plot_df.pivot(index='method', columns='minority_proportion', values=mean_col)
    
    # Sort proportions (columns) in descending order (balanced -> imbalanced)
    pivot = pivot[sorted(pivot.columns, reverse=True)]
    
    # Sort methods by average performance
    pivot['_mean'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('_mean', ascending=False)
    pivot = pivot.drop('_mean', axis=1)
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        cbar_kws={'label': metric},
        linewidths=0.5,
        ax=ax
    )
    
    ax.set_title(f'{task_name} Performance Heatmap: {metric} vs Minority Proportion{title_suffix}',
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Minority Class Proportion (Balanced -> Imbalanced)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Method', fontsize=12, fontweight='bold')
    
    # Format column labels as percentages
    ax.set_xticklabels([f'{x*100:.0f}%' for x in pivot.columns], rotation=45, ha='right')
    
    plt.tight_layout()
    
    # Save figure
    ds_suffix = f"_{dataset.replace('.', '_')}" if dataset else ""
    filename = FIGURES_DIR / f"{task_name.lower()}_heatmap_imbalance_{metric.lower()}{ds_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")
    
    return fig, pivot

In [ ]:
def calculate_imbalance_degradation(
    df: pd.DataFrame,
    metric: str,
    task_name: str
) -> pd.DataFrame:
    """
    Calculate performance degradation as class imbalance increases.
    
    Metrics computed:
    - perf_balanced: Performance at highest minority proportion (most balanced)
    - perf_imbalanced: Performance at lowest minority proportion (most imbalanced)
    - abs_drop: Absolute drop in performance
    - pct_drop: Percentage drop in performance
    - degradation_rate: Average drop per 1% decrease in minority proportion
    
    Args:
        df: Aggregated dataframe
        metric: Metric to analyze
        task_name: 'PD' or 'LGD'
    
    Returns:
        DataFrame with degradation metrics per method
    """
    mean_col = f'{metric}_mean'
    
    if mean_col not in df.columns:
        print(f"Warning: Metric '{metric}' not found")
        return pd.DataFrame()
    
    # Average across datasets first
    avg_df = df.groupby(['method', 'minority_proportion'])[mean_col].mean().reset_index()
    
    results = []
    
    for method in avg_df['method'].unique():
        method_df = avg_df[avg_df['method'] == method].sort_values('minority_proportion', ascending=False)
        
        prop_max = method_df['minority_proportion'].max()  # Most balanced
        prop_min = method_df['minority_proportion'].min()  # Most imbalanced
        
        perf_balanced = method_df[method_df['minority_proportion'] == prop_max][mean_col].values[0]
        perf_imbalanced = method_df[method_df['minority_proportion'] == prop_min][mean_col].values[0]
        
        abs_drop = perf_balanced - perf_imbalanced
        pct_drop = (abs_drop / perf_balanced * 100) if perf_balanced != 0 else 0
        
        # Degradation rate: performance drop per 1% decrease in minority proportion
        prop_diff = (prop_max - prop_min) * 100  # Convert to percentage points
        degradation_rate = (abs_drop / prop_diff) if prop_diff > 0 else 0
        
        results.append({
            'method': method,
            f'{metric}_balanced': perf_balanced,
            f'{metric}_imbalanced': perf_imbalanced,
            'prop_balanced': prop_max,
            'prop_imbalanced': prop_min,
            'abs_drop': abs_drop,
            'pct_drop': pct_drop,
            'degradation_rate_per_1pct': degradation_rate
        })
    
    result_df = pd.DataFrame(results)
    
    # Sort by performance at balanced state (best first)
    result_df = result_df.sort_values(f'{metric}_balanced', ascending=False)
    
    return result_df

In [ ]:
def plot_degradation_analysis(
    degradation_df: pd.DataFrame,
    metric: str,
    task_name: str,
    figsize: tuple = (16, 6)
):
    """
    Create visualization of performance degradation due to class imbalance.
    
    Shows:
    - Performance at balanced vs imbalanced state
    - Percentage drop
    - Degradation rate
    """
    if degradation_df.empty:
        return None
    
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    # Sort by performance at balanced for consistent ordering
    df = degradation_df.sort_values(f'{metric}_balanced', ascending=False)
    methods = df['method'].values
    
    # Plot 1: Performance at balanced vs imbalanced
    ax1 = axes[0]
    x = np.arange(len(methods))
    width = 0.35
    
    prop_balanced = df['prop_balanced'].iloc[0]
    prop_imbalanced = df['prop_imbalanced'].iloc[0]
    
    bars1 = ax1.bar(x - width/2, df[f'{metric}_balanced'], width, 
                    label=f'Balanced ({prop_balanced*100:.0f}%)', color='forestgreen', alpha=0.8)
    bars2 = ax1.bar(x + width/2, df[f'{metric}_imbalanced'], width, 
                    label=f'Imbalanced ({prop_imbalanced*100:.0f}%)', color='indianred', alpha=0.8)
    
    ax1.set_xticks(x)
    ax1.set_xticklabels(methods, rotation=45, ha='right')
    ax1.set_ylabel(metric, fontweight='bold')
    ax1.set_title(f'{metric}: Balanced vs Imbalanced', fontweight='bold')
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    
    # Plot 2: Percentage drop
    ax2 = axes[1]
    colors = ['forestgreen' if x < 10 else 'gold' if x < 25 else 'indianred' for x in df['pct_drop']]
    bars = ax2.bar(x, df['pct_drop'], color=colors, alpha=0.8, edgecolor='black')
    
    ax2.set_xticks(x)
    ax2.set_xticklabels(methods, rotation=45, ha='right')
    ax2.set_ylabel('Performance Drop (%)', fontweight='bold')
    ax2.set_title('Percentage Drop (Balanced -> Imbalanced)', fontweight='bold')
    ax2.axhline(y=10, color='green', linestyle='--', alpha=0.5, label='10% threshold')
    ax2.axhline(y=25, color='red', linestyle='--', alpha=0.5, label='25% threshold')
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(df['pct_drop']):
        ax2.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)
    
    # Plot 3: Degradation rate
    ax3 = axes[2]
    df_sorted = df.sort_values('degradation_rate_per_1pct', ascending=True)
    colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(df_sorted)))
    
    ax3.barh(df_sorted['method'], df_sorted['degradation_rate_per_1pct'], color=colors, alpha=0.8, edgecolor='black')
    ax3.set_xlabel(f'{metric} Drop per 1% Decrease in Minority Proportion', fontweight='bold')
    ax3.set_title('Degradation Rate (Lower = More Robust)', fontweight='bold')
    ax3.grid(axis='x', alpha=0.3)
    
    plt.suptitle(f'{task_name} Performance Degradation Under Class Imbalance', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    # Save figure
    filename = FIGURES_DIR / f"{task_name.lower()}_imbalance_degradation_{metric.lower()}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")
    
    return fig

In [ ]:
def plot_robustness_scatter(
    df: pd.DataFrame,
    metric: str,
    task_name: str,
    figsize: tuple = (12, 8)
):
    """
    Create a scatter plot showing performance vs robustness to imbalance.
    
    X-axis: Performance at balanced state (higher = better)
    Y-axis: Performance drop percentage (lower = more robust)
    
    Best methods are in the bottom-right quadrant.
    """
    degradation_df = calculate_imbalance_degradation(df, metric, task_name)
    
    if degradation_df.empty:
        return None
    
    fig, ax = plt.subplots(figsize=figsize)
    
    x = degradation_df[f'{metric}_balanced']
    y = degradation_df['pct_drop']
    methods = degradation_df['method']
    
    # Create scatter plot
    scatter = ax.scatter(x, y, s=100, c=range(len(methods)), cmap='viridis', alpha=0.7, edgecolors='black')
    
    # Add method labels
    for i, method in enumerate(methods):
        ax.annotate(method, (x.iloc[i], y.iloc[i]), 
                   xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    # Add quadrant lines
    ax.axhline(y=y.median(), color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=x.median(), color='gray', linestyle='--', alpha=0.5)
    
    # Add quadrant labels
    ax.text(x.max(), y.min(), 'BEST\n(High Perf, Robust)', ha='right', va='bottom', 
            fontsize=10, color='green', fontweight='bold', alpha=0.7)
    ax.text(x.min(), y.max(), 'WORST\n(Low Perf, Fragile)', ha='left', va='top',
            fontsize=10, color='red', fontweight='bold', alpha=0.7)
    
    ax.set_xlabel(f'{metric} at Balanced State', fontsize=12, fontweight='bold')
    ax.set_ylabel('Performance Drop Under Imbalance (%)', fontsize=12, fontweight='bold')
    ax.set_title(f'{task_name} Robustness vs Performance (Class Imbalance)',
                 fontsize=14, fontweight='bold', pad=15)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure
    filename = FIGURES_DIR / f"{task_name.lower()}_robustness_scatter_imbalance_{metric.lower()}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")
    
    return fig

In [ ]:
def plot_relative_performance(
    df: pd.DataFrame,
    metric: str,
    task_name: str,
    figsize: tuple = (14, 8)
):
    """
    Plot relative performance (normalized to performance at balanced state).
    
    This shows how much each method's performance drops relative to its best.
    A value of 1.0 means no drop, 0.9 means 10% drop.
    """
    mean_col = f'{metric}_mean'
    
    if mean_col not in df.columns:
        return None
    
    # Average across datasets
    avg_df = df.groupby(['method', 'minority_proportion'])[mean_col].mean().reset_index()
    
    # Calculate relative performance for each method
    relative_data = []
    
    for method in avg_df['method'].unique():
        method_df = avg_df[avg_df['method'] == method].sort_values('minority_proportion', ascending=False)
        max_perf = method_df[mean_col].iloc[0]  # Performance at highest proportion (most balanced)
        
        for _, row in method_df.iterrows():
            relative_perf = row[mean_col] / max_perf if max_perf != 0 else 0
            relative_data.append({
                'method': method,
                'minority_proportion': row['minority_proportion'],
                'relative_perf': relative_perf
            })
    
    rel_df = pd.DataFrame(relative_data)
    
    # Get top methods by performance at balanced
    max_prop = avg_df['minority_proportion'].max()
    top_methods = (
        avg_df[avg_df['minority_proportion'] == max_prop]
        .nlargest(12, mean_col)['method'].tolist()
    )
    
    rel_df = rel_df[rel_df['method'].isin(top_methods)]
    
    # Create plot
    fig, ax = plt.subplots(figsize=figsize)
    
    colors = sns.color_palette('husl', n_colors=len(top_methods))
    
    for i, method in enumerate(top_methods):
        method_df = rel_df[rel_df['method'] == method].sort_values('minority_proportion', ascending=False)
        ax.plot(method_df['minority_proportion'], method_df['relative_perf'], 
               'o-', label=method, color=colors[i], linewidth=2, markersize=6)
    
    ax.set_xlabel('Minority Class Proportion', fontsize=12, fontweight='bold')
    ax.set_ylabel('Relative Performance (1.0 = Best)', fontsize=12, fontweight='bold')
    ax.set_title(f'{task_name} Relative Performance Under Class Imbalance',
                 fontsize=14, fontweight='bold', pad=15)
    
    # Add reference lines
    ax.axhline(y=0.9, color='orange', linestyle='--', alpha=0.7, label='10% drop threshold')
    ax.axhline(y=0.75, color='red', linestyle='--', alpha=0.7, label='25% drop threshold')
    
    # Invert x-axis so imbalance increases left-to-right
    ax.invert_xaxis()
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))
    
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.5, 1.05)
    
    plt.tight_layout()
    
    filename = FIGURES_DIR / f"{task_name.lower()}_relative_performance_imbalance_{metric.lower()}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")
    
    return fig

In [ ]:
def create_rank_evolution_heatmap(
    df: pd.DataFrame,
    metric: str,
    task_name: str,
    figsize: tuple = (16, 10)
):
    """
    Create a heatmap showing how method RANKS change across minority proportions.
    
    This reveals which methods maintain their relative position vs which
    become better or worse as imbalance increases.
    """
    mean_col = f'{metric}_mean'
    
    if mean_col not in df.columns:
        return None
    
    # Average across datasets
    avg_df = df.groupby(['method', 'minority_proportion'])[mean_col].mean().reset_index()
    
    # Create pivot table
    pivot = avg_df.pivot(index='method', columns='minority_proportion', values=mean_col)
    
    # Calculate ranks (1 = best)
    rank_pivot = pivot.rank(ascending=False, method='min')
    
    # Sort columns (proportions) descending (balanced -> imbalanced)
    rank_pivot = rank_pivot[sorted(rank_pivot.columns, reverse=True)]
    
    # Sort methods by rank at highest proportion (most balanced)
    max_col = rank_pivot.columns[0]
    rank_pivot = rank_pivot.sort_values(max_col)
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    sns.heatmap(
        rank_pivot,
        annot=True,
        fmt='.0f',
        cmap='RdYlGn_r',  # Reversed: green=rank 1, red=high rank
        cbar_kws={'label': 'Rank (1=Best)'},
        linewidths=0.5,
        ax=ax
    )
    
    ax.set_title(f'{task_name} Rank Evolution: {metric} Across Minority Proportions',
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Minority Proportion (Balanced -> Imbalanced)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Method', fontsize=12, fontweight='bold')
    
    # Format x-axis labels as percentages
    ax.set_xticklabels([f'{x*100:.0f}%' for x in rank_pivot.columns], rotation=45, ha='right')
    
    plt.tight_layout()
    
    filename = FIGURES_DIR / f"{task_name.lower()}_rank_evolution_imbalance_{metric.lower()}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")
    
    return fig, rank_pivot

In [ ]:
def plot_metric_comparison_imbalance(
    df: pd.DataFrame,
    metrics: list,
    task_name: str,
    method: str = None,
    figsize: tuple = (16, 5)
):
    """
    Compare multiple metrics across imbalance levels.
    
    This is particularly useful to see which metrics are more sensitive
    to class imbalance (e.g., Recall/F1 often drop more than AUC).
    
    Args:
        df: Aggregated dataframe
        metrics: List of metrics to compare (e.g., ['AUC', 'F1', 'Recall'])
        task_name: 'PD' or 'LGD'
        method: Specific method to plot (if None, averages across all methods)
        figsize: Figure size
    """
    # Prepare data
    plot_df = df.copy()
    if method is not None:
        plot_df = plot_df[plot_df['method'] == method]
        title_suffix = f" - {method}"
    else:
        # Average across all methods
        agg_dict = {f'{m}_mean': 'mean' for m in metrics if f'{m}_mean' in df.columns}
        plot_df = plot_df.groupby('minority_proportion').agg(agg_dict).reset_index()
        title_suffix = " (Averaged Across Methods)"
    
    fig, axes = plt.subplots(1, len(metrics), figsize=figsize)
    
    if len(metrics) == 1:
        axes = [axes]
    
    for i, metric in enumerate(metrics):
        mean_col = f'{metric}_mean'
        if mean_col not in plot_df.columns:
            continue
        
        ax = axes[i]
        
        data = plot_df.sort_values('minority_proportion', ascending=False)
        x = data['minority_proportion'].values
        y = data[mean_col].values
        
        # Normalize to 0-1 for comparison
        y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() != y.min() else y
        
        ax.plot(x, y, 'o-', linewidth=2, markersize=8, color=IMBALANCE_CURVE_PALETTE[i])
        ax.fill_between(x, 0, y, alpha=0.2, color=IMBALANCE_CURVE_PALETTE[i])
        
        ax.set_xlabel('Minority Proportion', fontweight='bold')
        ax.set_ylabel(metric, fontweight='bold')
        ax.set_title(f'{metric}', fontweight='bold')
        ax.invert_xaxis()
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x*100:.0f}%'))
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'{task_name} Metric Comparison Under Class Imbalance{title_suffix}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    # Save figure
    method_suffix = f"_{method}" if method else ""
    filename = FIGURES_DIR / f"{task_name.lower()}_metric_comparison_imbalance{method_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")
    
    return fig

---

# Part A: PD (Probability of Default) - Imbalance Analysis

---

## A1. Imbalance Curve Visualizations

### A1.1 Overall Imbalance Curves (AUC)

In [ ]:
if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    print("Creating PD AUC Imbalance Curves (Top 15 Methods)...")
    plot_imbalance_curves(pd_agg, 'AUC', 'PD', top_n=15)
    plt.show()
else:
    print("No PD class imbalance data available")

### A1.2 Overall Imbalance Curves (F1)

In [ ]:
if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    print("Creating PD F1 Imbalance Curves (Top 15 Methods)...")
    plot_imbalance_curves(pd_agg, 'F1', 'PD', top_n=15)
    plt.show()

### A1.3 Overall Imbalance Curves (Recall)

Recall is particularly important for credit risk - we want to catch all defaults!

In [ ]:
if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    print("Creating PD Recall Imbalance Curves (Top 15 Methods)...")
    plot_imbalance_curves(pd_agg, 'Recall', 'PD', top_n=15)
    plt.show()

### A1.4 Relative Performance Decay

In [ ]:
if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    print("Creating PD Relative Performance Decay...")
    plot_relative_performance(pd_agg, 'AUC', 'PD')
    plt.show()

## A2. Performance Heatmaps

### A2.1 Performance Heatmap (Methods x Minority Proportions)

In [ ]:
if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    print("Creating PD AUC Performance Heatmap...")
    fig, pivot = plot_imbalance_heatmap(pd_agg, 'AUC', 'PD')
    plt.show()

### A2.2 Rank Evolution Heatmap

In [ ]:
if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    print("Creating PD Rank Evolution Heatmap...")
    fig, rank_pivot = create_rank_evolution_heatmap(pd_agg, 'AUC', 'PD')
    plt.show()
    
    print("\nMethods that improve rank as imbalance increases (potential imbalance specialists):")
    max_col = rank_pivot.columns[0]  # Most balanced
    min_col = rank_pivot.columns[-1]  # Most imbalanced
    rank_change = rank_pivot[max_col] - rank_pivot[min_col]
    improvers = rank_change[rank_change > 2].sort_values(ascending=False)
    if not improvers.empty:
        print(improvers)
    else:
        print("No significant rank improvements found.")

## A3. Degradation Analysis

### A3.1 Degradation Metrics Table

In [ ]:
if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    pd_degradation = calculate_imbalance_degradation(pd_agg, 'AUC', 'PD')
    
    print("\n" + "=" * 80)
    print("  PD DEGRADATION ANALYSIS (AUC) - CLASS IMBALANCE")
    print("=" * 80)
    print("\nMethods sorted by performance at balanced state:")
    display(pd_degradation.round(4))
    
    print("\n" + "-" * 40)
    print("Most ROBUST methods (lowest % drop):")
    print("-" * 40)
    print(pd_degradation.nsmallest(5, 'pct_drop')[['method', 'AUC_balanced', 'AUC_imbalanced', 'pct_drop']].to_string(index=False))
    
    print("\n" + "-" * 40)
    print("LEAST robust methods (highest % drop):")
    print("-" * 40)
    print(pd_degradation.nlargest(5, 'pct_drop')[['method', 'AUC_balanced', 'AUC_imbalanced', 'pct_drop']].to_string(index=False))

### A3.2 Degradation Visualization

In [ ]:
if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    print("Creating PD Degradation Analysis Plots...")
    plot_degradation_analysis(pd_degradation, 'AUC', 'PD')
    plt.show()

### A3.3 Robustness vs Performance Scatter

In [ ]:
if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    print("Creating PD Robustness vs Performance Scatter...")
    plot_robustness_scatter(pd_agg, 'AUC', 'PD')
    plt.show()

## A4. Metric Comparison Under Imbalance

Different metrics have different sensitivities to class imbalance. This section compares how AUC, F1, Recall, and Precision behave.

In [ ]:
if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    print("Creating Metric Comparison Across Imbalance Levels...")
    plot_metric_comparison_imbalance(pd_agg, ['AUC', 'F1', 'Recall', 'Precision'], 'PD')
    plt.show()

## A5. F1 Score Deep Dive

F1 is often more sensitive to class imbalance than AUC. Let's analyze it in detail.

In [ ]:
if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    f1_degradation = calculate_imbalance_degradation(pd_agg, 'F1', 'PD')
    
    print("\n" + "=" * 80)
    print("  PD DEGRADATION ANALYSIS (F1) - CLASS IMBALANCE")
    print("=" * 80)
    print("\nMethods sorted by performance at balanced state:")
    display(f1_degradation.round(4))
    
    print("\nCreating F1 Degradation Analysis Plots...")
    plot_degradation_analysis(f1_degradation, 'F1', 'PD')
    plt.show()

## A6. Per-Dataset Analysis

In [ ]:
if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    # Show imbalance curves for first 3 datasets
    datasets = sorted(pd_agg['dataset'].unique())[:3]
    
    for dataset in datasets:
        print(f"\nImbalance Curve for Dataset: {dataset}")
        plot_imbalance_curves(pd_agg, 'AUC', 'PD', dataset=dataset, top_n=10)
        plt.show()

---

# Part B: Summary and Recommendations

---

## B1. Method Recommendations by Imbalance Level

In [ ]:
def create_recommendations_by_imbalance(df, metric, task_name):
    """
    Create a recommendations table showing best methods at different imbalance levels.
    """
    mean_col = f'{metric}_mean'
    
    if mean_col not in df.columns:
        return None
    
    # Average across datasets
    avg_df = df.groupby(['method', 'minority_proportion'])[mean_col].mean().reset_index()
    
    # Get unique proportions
    proportions = sorted(avg_df['minority_proportion'].unique(), reverse=True)
    
    recommendations = []
    
    for prop in proportions:
        subset = avg_df[avg_df['minority_proportion'] == prop].nlargest(5, mean_col)
        rec = {
            'Minority %': f"{prop*100:.0f}%",
            'Imbalance Level': 'Balanced' if prop > 0.4 else ('Moderate' if prop > 0.1 else 'Severe'),
            '1st': f"{subset.iloc[0]['method']} ({subset.iloc[0][mean_col]:.4f})",
            '2nd': f"{subset.iloc[1]['method']} ({subset.iloc[1][mean_col]:.4f})" if len(subset) > 1 else '-',
            '3rd': f"{subset.iloc[2]['method']} ({subset.iloc[2][mean_col]:.4f})" if len(subset) > 2 else '-',
        }
        recommendations.append(rec)
    
    rec_df = pd.DataFrame(recommendations)
    
    print(f"\n{'='*80}")
    print(f"  {task_name} TOP METHODS BY IMBALANCE LEVEL ({metric})")
    print(f"{'='*80}")
    display(rec_df)
    
    return rec_df

if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    pd_recommendations = create_recommendations_by_imbalance(pd_agg, 'AUC', 'PD')

## B2. Final Summary Statistics

In [ ]:
print("\n" + "=" * 80)
print("  EXPERIMENT 3: CLASS IMBALANCE ANALYSIS SUMMARY")
print("=" * 80)

if has_pd and not pd_agg.empty and 'minority_proportion' in pd_agg.columns:
    proportions = sorted(pd_agg['minority_proportion'].unique(), reverse=True)
    print(f"\nPD Task (Classification):")
    print(f"  - Methods analyzed: {pd_agg['method'].nunique()}")
    print(f"  - Datasets: {pd_agg['dataset'].nunique()}")
    print(f"  - Minority proportion range: {max(proportions)*100:.0f}% -> {min(proportions)*100:.0f}%")
    print(f"  - Number of imbalance levels: {pd_agg['minority_proportion'].nunique()}")

print(f"\nFigures saved to: {FIGURES_DIR}")
print("=" * 80)

## B3. Key Insights

**Interpretation Guide:**

1. **Imbalance Curves** show how each method's performance changes with minority class proportion.
   - Flat curves = robust to class imbalance
   - Steep curves = sensitive to imbalance

2. **Degradation Analysis** quantifies the performance drop:
   - `pct_drop` < 10%: Very robust to imbalance
   - `pct_drop` 10-25%: Moderately affected
   - `pct_drop` > 25%: Highly sensitive to imbalance

3. **Robustness vs Performance Scatter**:
   - Bottom-right quadrant: Best overall (high performance, robust to imbalance)
   - Top-right: Good performance but sensitive to imbalance
   - Bottom-left: Robust but lower performance

4. **Rank Evolution** shows competitive dynamics:
   - Methods that improve rank with more imbalance are "imbalance specialists"
   - Methods that drop rank need balanced data to shine

**For Method Selection in Credit Risk:**

- **Highly Imbalanced Datasets (< 5% default rate)**: Prioritize robust methods
- **Moderately Imbalanced (5-20% default rate)**: More flexibility in method choice
- **Nearly Balanced (> 20%)**: Can use any well-performing method

**Important Metrics to Watch:**

- **AUC**: Generally stable under imbalance (threshold-independent)
- **F1 Score**: More sensitive to imbalance (depends on threshold)
- **Recall**: Critical for credit risk - can degrade significantly
- **Precision**: Often improves as minority class shrinks (trivial predictions)

**Practical Recommendations:**

1. For production credit scoring with severe imbalance, choose methods from the "robust" category
2. Consider class weights or oversampling if your preferred method is imbalance-sensitive
3. Always evaluate with multiple metrics - AUC alone can be misleading